# LearnSync AI — Member 2: RAG and Text Analytics
### Hybrid Agentic Personalized Learning Tutor with Dynamic Curriculum Adaptation
**Role:** Member 2 — RAG Content Agent & Answer Evaluation Agent  
**Domain:** Data Structures and Algorithms (DSA)  
**Target Environment:** Google Colab / Python 3.10+  

---
### Notebook Workflow
1. **Environment & Dependency Setup** (`sentence-transformers`, `faiss-cpu`, etc.)
2. **OpenDSA Content Ingestion & Semantic Chunking** (`opendsa_chunks.csv`)
3. **Vector Index Construction** (Hugging Face embeddings + FAISS index)
4. **Retrieval Pipeline & Relevance Evaluation** (Hit Rate@k, MRR, Latency)
5. **Mastery-Conditioned Grounded Explanation Generation**
6. **Short-Answer Benchmark Dataset Exploration** (`short_answer_dataset.csv`)
7. **Rule-Based Answer Evaluation & Misconception Detection Engine**
8. **Quantitative Evaluation & Confusion Matrix**
9. **Integration Module Export & Contract Validation**

## 1. Environment & Dependency Setup
Install required libraries for Hugging Face Sentence Transformers and FAISS vector indexing.

In [1]:
# Uncomment in Google Colab:
# !pip install -q sentence-transformers faiss-cpu pandas scikit-learn

import os
import csv
import json
import time
import numpy as np
print('Environment initialized successfully.')

Environment initialized successfully.


## 2. OpenDSA Content Ingestion & Semantic Chunk Inspection
Load and inspect the curated OpenDSA instructional chunks covering all 13 standard concepts.

In [2]:
from rag import load_opendsa_content, VALID_CONCEPTS

chunks = load_opendsa_content('opendsa_chunks.csv')
print(f'Loaded {len(chunks)} semantic chunks from OpenDSA.\n')

concept_counts = {}
for c in chunks:
    cid = c['concept_id']
    concept_counts[cid] = concept_counts.get(cid, 0) + 1

print('Chunks per DSA Concept:')
for cid, count in sorted(concept_counts.items()):
    print(f' - {cid:<22}: {count} chunks')

print('\nSample Chunk Metadata:')
sample = chunks[0]
print(json.dumps({
    'chunk_id': sample['chunk_id'],
    'concept_id': sample['concept_id'],
    'title': sample['title'],
    'source_url': sample['source_url'],
    'difficulty_level': sample['difficulty_level'],
    'summary': sample['summary']
}, indent=2))

Loaded 39 semantic chunks from OpenDSA.

Chunks per DSA Concept:
 - arrays                : 3 chunks
 - bfs                   : 3 chunks
 - binary_search_trees   : 3 chunks
 - dfs                   : 3 chunks
 - graphs                : 3 chunks
 - hashing               : 3 chunks
 - heaps                 : 3 chunks
 - linked_lists          : 3 chunks
 - queues                : 3 chunks
 - recursion             : 3 chunks
 - sorting               : 3 chunks
 - stacks                : 3 chunks
 - trees                 : 3 chunks

Sample Chunk Metadata:
{
  "chunk_id": "chk_arr_01",
  "concept_id": "arrays",
  "title": "Array Data Structure and Random Access",
  "source_url": "https://opendsa-server.cs.vt.edu/ODSA/Books/CS3/html/ListArray.html",
  "difficulty_level": "beginner",
  "summary": "Arrays store elements in contiguous memory enabling O(1) index access with O(n) arbitrary insertion or deletion."
}


## 3. Vector Index Construction
Build vector search index with dense representations and cosine similarity.

In [3]:
from rag import build_vector_index

index, indexed_docs = build_vector_index(chunks)
print(f'Vector index successfully built with {len(indexed_docs)} documents.')

c:\Users\kkris\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2242.71it/s]


Vector index successfully built with 39 documents.


## 4. Retrieval Pipeline & Empirical Relevance Evaluation
Evaluate retrieval effectiveness across 13 distinct queries, calculating Hit Rate@1, Hit Rate@3, and MRR.

The 13 queries in this notebook are a demonstration benchmark. The authoritative retrieval benchmark is `test_rag_pipeline.py`, which evaluates 26 queries across all 13 concepts.

In [4]:
from rag import retrieve_context

benchmark_queries = [
    ('How does an array store elements in contiguous memory?', 'arrays', 'chk_arr_01'),
    ('Explain pointer based traversal in linked lists', 'linked_lists', 'chk_ll_01'),
    ('What is the LIFO principle in stacks?', 'stacks', 'chk_stk_01'),
    ('How do queues implement FIFO ordering?', 'queues', 'chk_que_01'),
    ('Why are base cases essential for recursion?', 'recursion', 'chk_rec_01'),
    ('What are tree traversals like preorder and inorder?', 'trees', 'chk_tre_02'),
    ('Explain the binary search tree property', 'binary_search_trees', 'chk_bst_01'),
    ('What is a max heap and priority queue?', 'heaps', 'chk_hea_01'),
    ('Compare merge sort and quicksort', 'sorting', 'chk_srt_02'),
    ('How do hash tables resolve collisions with chaining?', 'hashing', 'chk_has_02'),
    ('What is the difference between adjacency list and matrix?', 'graphs', 'chk_grp_01'),
    ('How does breadth first search find shortest paths?', 'bfs', 'chk_bfs_01'),
    ('Explain depth first search and backtracking', 'dfs', 'chk_dfs_01')
]

hits_at_1 = 0
hits_at_3 = 0
rr_list = []

print(f"{'Query':<45} | {'Concept':<18} | {'Rank':<6} | {'Top Chunk'}")
print('-' * 85)
for q, concept, target in benchmark_queries:
    retrieved = retrieve_context(q, concept_id=concept, top_k=3)
    c_ids = [item['chunk_id'] for item in retrieved]
    top_id = c_ids[0] if c_ids else 'None'
    rank = c_ids.index(target) + 1 if target in c_ids else None
    
    if rank == 1:
        hits_at_1 += 1
    if rank and rank <= 3:
        hits_at_3 += 1
        rr_list.append(1.0 / rank)
    else:
        rr_list.append(0.0)
        
    print(f"{q[:43]:<45} | {concept:<18} | {str(rank):<6} | {top_id}")

n_q = len(benchmark_queries)
print('\n--- Retrieval Evaluation Summary ---')
print(f'Hit Rate@1: {hits_at_1}/{n_q} ({hits_at_1/n_q*100:.1f}%)')
print(f'Hit Rate@3: {hits_at_3}/{n_q} ({hits_at_3/n_q*100:.1f}%)')
print(f'Mean Reciprocal Rank (MRR): {sum(rr_list)/n_q:.4f}')

Query                                         | Concept            | Rank   | Top Chunk
-------------------------------------------------------------------------------------
How does an array store elements in contigu   | arrays             | 1      | chk_arr_01
Explain pointer based traversal in linked l   | linked_lists       | 1      | chk_ll_01
What is the LIFO principle in stacks?         | stacks             | 1      | chk_stk_01
How do queues implement FIFO ordering?        | queues             | 1      | chk_que_01
Why are base cases essential for recursion?   | recursion          | 1      | chk_rec_01
What are tree traversals like preorder and    | trees              | 1      | chk_tre_02
Explain the binary search tree property       | binary_search_trees | 1      | chk_bst_01
What is a max heap and priority queue?        | heaps              | 1      | chk_hea_01
Compare merge sort and quicksort              | sorting            | 1      | chk_srt_02
How do hash tables resolv

## 5. Grounded Explanation Generation Conditioned on Mastery
Demonstrate pedagogical personalization based on student BKT mastery estimate.

In [5]:
from rag import generate_explanation

query = 'Explain recursion and why base cases are required.'
context = retrieve_context(query, concept_id='recursion', top_k=2)

print('=== REMEDIAL EXPLANATION (Mastery = 0.20) ===')
print(generate_explanation(query, context, mastery_level=0.20))

print('\n=== ADVANCED EXPLANATION (Mastery = 0.85) ===')
print(generate_explanation(query, context, mastery_level=0.85))

=== REMEDIAL EXPLANATION (Mastery = 0.20) ===
> *[Mode: Deterministic Offline Fallback (Reason: API key not configured)]*

## Instructional Response: Explain recursion and why base cases are required.

**Pedagogical Target:** Foundational Tier (Mastery: 20.0%)

### 🟢 Core Concept Fundamentals & Summary

**Topic:** Recursion Principles and Base Cases

**Core Principle:** Recursion requires terminating base cases and recursive steps that strictly reduce toward base cases.

**Reference Passage:**
Recursion is an algorithmic problem-solving approach where a function invokes itself with smaller instances of the identical problem. Every valid recursive algorithm requires two components: at least one base case that terminates recursion without further self-calls, and a recursive step that shrinks input parameters strictly toward the base case. Omitting a base case causes infinite recursion and runtime stack overflow.

**Key Takeaway:** Begin by understanding the fundamental operational rules 

## 6. Short-Answer Dataset & Misconception Inspection
Load `short_answer_dataset.csv` and inspect test cases.

In [6]:
with open('short_answer_dataset.csv', 'r', encoding='utf-8') as f:
    qa_data = list(csv.DictReader(f))

print(f'Loaded {len(qa_data)} student short-answer test pairs.')
class_counts = {}
for row in qa_data:
    c = row['ground_truth_classification']
    class_counts[c] = class_counts.get(c, 0) + 1

print('Ground Truth Label Distribution:', class_counts)

Loaded 117 student short-answer test pairs.
Ground Truth Label Distribution: {'correct': 39, 'partially_correct': 39, 'incorrect': 39}


## 7. Rule-Based Answer Evaluation & Misconception Detection
Evaluate student natural language responses using `evaluate_answer()`. The evaluator is a rule-based classifier based on regex rubrics, invariant checks, and negation handling. SentenceTransformer similarity is logged as supporting evidence only.

In [7]:
from evaluator import evaluate_answer

# Example 1: Student answer with a misconception
misc_answer = 'A stack is First-In First-Out (FIFO) like a supermarket queue. Push is O(1) and pop is O(n).'
eval_res = evaluate_answer('Q_STK_01', misc_answer)  # Canonical ID (legacy alias: Q_STACK_01)
print('Evaluator Output (Misconception Detected):')
print(json.dumps(eval_res, indent=2))

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1398.07it/s]


Evaluator Output (Misconception Detected):
{
  "concept_id": "stacks",
  "classification": "incorrect",
  "score": 0.0,
  "bkt_outcome": 0,
  "identified_misconceptions": [
    "Confuses LIFO with FIFO"
  ],
  "feedback": "A stack strictly operates under the Last-In First-Out (LIFO) rule; FIFO governs queues.",
  "evidence": [
    "Detected conceptual error: 'First-In First-Out'",
    "Supporting evidence (cosine similarity to reference): 0.7914"
  ]
}


## 8. Quantitative Evaluation & Confusion Matrix
Benchmark the evaluator across the complete short-answer dataset.

In [8]:
classes = ['correct', 'partially_correct', 'incorrect']
cm = {gt: {pred: 0 for pred in classes} for gt in classes}
correct_evals = 0
misc_detected = 0
misc_total = 0

for row in qa_data:
    qid = row['question_id']
    ans = row['student_answer']
    gt = row['ground_truth_classification']
    exp_misc = row.get('expected_misconceptions', '')
    
    res = evaluate_answer(qid, ans)
    pred = res['classification']
    cm[gt][pred] += 1
    if pred == gt:
        correct_evals += 1
        
    if exp_misc not in ('', 'None', None):
        misc_total += 1
        if exp_misc in res['identified_misconceptions']:
            misc_detected += 1

print(f'Answer Classification Accuracy: {correct_evals}/{len(qa_data)} = {correct_evals/len(qa_data)*100:.2f}%')
print(f'Exact-Label Misconception Match Rate: {misc_detected}/{misc_total} = {misc_detected/misc_total*100:.2f}%\n')

print('Confusion Matrix:')
print(f"{'GT / Pred':<18} | {'correct':<10} | {'partially':<10} | {'incorrect':<10}")
print('-' * 55)
for gt in classes:
    print(f"{gt:<18} | {cm[gt]['correct']:^10} | {cm[gt]['partially_correct']:^10} | {cm[gt]['incorrect']:^10}")

Answer Classification Accuracy: 117/117 = 100.00%
Exact-Label Misconception Match Rate: 55/78 = 70.51%

Confusion Matrix:
GT / Pred          | correct    | partially  | incorrect 
-------------------------------------------------------
correct            |     39     |     0      |     0     
partially_correct  |     0      |     39     |     0     
incorrect          |     0      |     0      |     39    


## 9. Integration Contract Verification
Verify that imports match Part C Integration Rules and test BKT conversion.

In [9]:
from rag import retrieve_context, generate_explanation
from evaluator import evaluate_answer, bkt_outcome_from_evaluation

# Verify required interfaces and BKT binary outcome conversion
test_eval = evaluate_answer('Q_BST_01', 'In a BST left subtree is smaller and right subtree is greater.')
outcome = bkt_outcome_from_evaluation(test_eval)
print('Contract Import & Execution Test:')
print(f"Score: {test_eval['score']}, Classification: {test_eval['classification']}, BKT Outcome: {outcome}")
print('\nAll Member 2 deliverables verified and ready for orchestration integration!')

Contract Import & Execution Test:
Score: 0.5, Classification: partially_correct, BKT Outcome: 0

All Member 2 deliverables verified and ready for orchestration integration!


In [10]:
from run_heldout_eval import evaluate_heldout_dataset

evaluate_heldout_dataset()

LearnSync AI - Held-Out Benchmark Evaluation (heldout_answers.csv, 26 Rows)
Note: The 26 held-out examples were not used during rule development.
This evaluation measures out-of-sample generalization of the rule-based classifier.

Total Held-Out Samples: 26
Overall Classification Accuracy: 9/26 (34.62%)

3-Class Confusion Matrix:
Actual \ Predicted     | correct    | partially_correct  | incorrect 
----------------------------------------------------------------------
correct                | 4          | 8                  | 0         
partially_correct      | 2          | 5                  | 0         
incorrect              | 2          | 5                  | 0         

Per-Class Classification Metrics:
Class                | Precision  | Recall     | F1-Score   | Support 
----------------------------------------------------------------------
correct              | 0.5000     | 0.3333     | 0.4000     | 12      
partially_correct    | 0.2778     | 0.7143     | 0.4000     | 7      

{'accuracy': 34.61538461538461,
 'confusion_matrix': {'correct': {'correct': 4,
   'partially_correct': 8,
   'incorrect': 0},
  'partially_correct': {'correct': 2, 'partially_correct': 5, 'incorrect': 0},
  'incorrect': {'correct': 2, 'partially_correct': 5, 'incorrect': 0}},
 'metrics_per_class': {'correct': {'precision': 0.5,
   'recall': 0.3333333333333333,
   'f1': 0.4,
   'support': 12},
  'partially_correct': {'precision': 0.2777777777777778,
   'recall': 0.7142857142857143,
   'f1': 0.4,
   'support': 7},
  'incorrect': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 7}},
 'macro_f1': 0.26666666666666666,
 'weighted_f1': 0.2923076923076924,
 'misclassified': [{'row_index': 1,
   'question_id': 'Q_ARR_01',
   'student_answer': 'The items are stored next to each other, so the computer can jump straight to an index. If I add something at the start, the other items have to move along.',
   'actual_label': 'correct',
   'predicted_label': 'partially_correct',
   'score': 0.5